[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C37_MLOps_Course/01_experiment_tracking/01_experiment_tracking.ipynb)

# 01 · 从零实现实验追踪器

本 notebook 从零造一个「小 MLflow」：一个能 **记录 params/metrics/artifacts、管理 run 生命周期、比较与选最优、保证可复现** 的实验追踪器，用 **标准库（`json`/`hashlib`/`os`/`tempfile`）+ numpy** 实现，落真盘、过 `assert`。

**路线**：① run 数据模型与内容哈希 → ② 落盘的 Tracker（start/log/end）→ ③ 记录不可变性 → ④ 比较与选最优 → ⑤ config hash 复现校验 → ✏️ 4 道练习 → 📖 答案 → 🧪 用真实 `sqlite3` 建查询索引。

In [ ]:
import os, json, hashlib, tempfile, time
import numpy as np
rng = np.random.default_rng(0)
print('环境就绪 ✅  | numpy', np.__version__)

## 1 · run 数据模型与内容哈希

一次实验 = 一个 `run`：把 **输入（params）→ 输出（metrics）+ 产物（artifacts）+ 元数据（tags）** 结构化记下来。

先实现两个地基函数：`canonical_hash`（规范化 JSON 后哈希，键序无关）和 `new_run`（造一个空 run 骨架）。

In [ ]:
def canonical_hash(obj):
    '''内容哈希：sort_keys 保证键序无关，相同内容必得相同指纹。'''
    blob = json.dumps(obj, sort_keys=True, ensure_ascii=False).encode('utf-8')
    return hashlib.sha256(blob).hexdigest()

def new_run(params, tags=None):
    rid = canonical_hash({'params': params, 'tags': tags or {}, 't': time.time(), 'nonce': rng.integers(1<<30).item()})[:12]
    return {'run_id': rid, 'status': 'running', 'params': dict(params),
            'metrics': {}, 'history': {}, 'artifacts': {}, 'tags': dict(tags or {}),
            'start_ts': time.time(), 'end_ts': None}

r = new_run({'lr': 0.01, 'layers': 3, 'seed': 42}, tags={'owner': 'zerui'})
print('run_id =', r['run_id'], '| status =', r['status'])
# 不变量：相同配置内容、不同键序 -> 同哈希；改一点 -> 变
assert canonical_hash({'a':1,'b':2}) == canonical_hash({'b':2,'a':1})
assert canonical_hash({'a':1,'b':2}) != canonical_hash({'a':1,'b':3})
assert len(r['run_id']) == 12 and r['status'] == 'running'
print('✅ run 骨架与内容哈希正确（键序无关、对改动敏感）')

## 2 · 落盘的 Tracker：start_run / log_metric / log_artifact / end_run

把 run 持久化成 `root/<experiment>/<run_id>.json`（真写盘）。这就是 MLflow 本地后端的最小内核：一棵 append-only 的文件树。

`log_metric` 同时维护 **最终值**（用于选优）与 **带 step 的曲线**（用于诊断）；`log_artifact` 顺手算 **内容哈希**（产物可校验、接模块 02）。

In [ ]:
class Tracker:
    def __init__(self, root, experiment):
        self.dir = os.path.join(root, experiment)
        os.makedirs(self.dir, exist_ok=True)

    def _path(self, run):
        return os.path.join(self.dir, run['run_id'] + '.json')

    def _save(self, run):
        with open(self._path(run), 'w', encoding='utf-8') as f:
            json.dump(run, f, ensure_ascii=False, indent=1)

    def start_run(self, params, tags=None):
        run = new_run(params, tags)
        self._save(run)
        return run

    def log_metric(self, run, key, value, step=None):
        assert run['status'] == 'running', '不能向已结束的 run 记录指标（记录不可变）'
        run['metrics'][key] = float(value)                       # 最终值
        run['history'].setdefault(key, []).append([step, float(value)])  # 曲线
        self._save(run)

    def log_artifact(self, run, name, data_bytes):
        assert run['status'] == 'running'
        h = hashlib.sha256(data_bytes).hexdigest()
        run['artifacts'][name] = 'sha256:' + h
        self._save(run)
        return h

    def end_run(self, run, status='finished'):
        run['status'] = status
        run['end_ts'] = time.time()
        self._save(run)

    def load(self, run_id):
        with open(os.path.join(self.dir, run_id + '.json'), encoding='utf-8') as f:
            return json.load(f)

    def all_runs(self):
        out = []
        for fn in sorted(os.listdir(self.dir)):
            if fn.endswith('.json'):
                out.append(self.load(fn[:-5]))
        return out

print('Tracker 定义完成 ✅')

In [ ]:
# 跑一个真实的小实验并落盘
WORK = tempfile.mkdtemp()                       # 隔离的真实目录
tk = Tracker(WORK, 'demo')
run = tk.start_run({'lr': 0.01, 'layers': 5, 'seed': 42, 'data': 'v1'}, tags={'owner': 'zerui'})
for step, acc in enumerate([0.70, 0.85, 0.91, 0.913]):
    tk.log_metric(run, 'val_acc', acc, step=step)
h = tk.log_artifact(run, 'model.npz', b'\x93NUMPY fake weights')
tk.end_run(run, 'finished')

reloaded = tk.load(run['run_id'])              # 从盘上读回
print('落盘文件：', os.path.basename(tk._path(run)))
print('最终 val_acc =', reloaded['metrics']['val_acc'], '| 曲线点数 =', len(reloaded['history']['val_acc']))
print('产物哈希 =', reloaded['artifacts']['model.npz'][:23], '...')
assert reloaded['status'] == 'finished'
assert reloaded['metrics']['val_acc'] == 0.913
assert len(reloaded['history']['val_acc']) == 4         # 4 步曲线都在
assert reloaded['artifacts']['model.npz'] == 'sha256:' + h
print('✅ start→log→end 全程落盘、读回一致')

## 3 · 记录不可变性：封存后不能再改

一个 **finished** 的 run 必须冻结——否则「重跑一下顺便更新指标」会销毁历史，三个月后没人知道当初上线的到底是哪个数。

我们把这条纪律编码进 API：向已结束的 run `log_metric` 会直接抛 `AssertionError`。

In [ ]:
# run 已 finished，再记指标应当被拒
raised = False
try:
    tk.log_metric(run, 'val_acc', 0.999)        # 企图篡改历史
except AssertionError as e:
    raised = True
    print('被正确拒绝：', e)
assert raised, 'finished 的 run 必须拒绝再次写入'

# 想「再实验一次」？正确做法是开一个新 run（哪怕配置相同），历史保持不可变
run2 = tk.start_run({'lr': 0.01, 'layers': 5, 'seed': 42, 'data': 'v1'})
assert run2['run_id'] != run['run_id'], '新实验必须是新 run'
tk.end_run(run2, 'finished')
print('✅ 记录不可变：篡改被拒，重实验另开新 run', run2['run_id'])

## 4 · 比较与选最优：从记录到决策

追踪的终点是 **决策**：在一堆 run 里挑该上线的那个。纪律：① 只比 `finished`；② 统一指标口径；③ 给出可复现的最优 run。

先灌入几个 run（含一个 failed 的干扰项），再实现 `best_run`。

In [ ]:
# 造一批 run：不同超参，含一个 failed
specs = [
    ({'lr':0.01,'layers':3,'seed':42}, 0.901, 'finished'),
    ({'lr':0.05,'layers':3,'seed':42}, 0.890, 'finished'),
    ({'lr':0.01,'layers':5,'seed':42}, 0.918, 'finished'),   # 真最优
    ({'lr':0.10,'layers':3,'seed':42}, 0.640, 'failed'),     # 崩了，必须排除
    ({'lr':0.01,'layers':5,'seed':99}, 0.915, 'finished'),
]
tk2 = Tracker(WORK, 'tune')
for params, acc, st in specs:
    rr = tk2.start_run(params)
    tk2.log_metric(rr, 'val_acc', acc, step=0)
    tk2.end_run(rr, st)

def best_run(runs, metric, mode='max'):
    cand = [r for r in runs if r['status'] == 'finished' and metric in r['metrics']]
    assert cand, '没有可比较的 finished run'
    key = (lambda r: r['metrics'][metric])
    return max(cand, key=key) if mode == 'max' else min(cand, key=key)

runs = tk2.all_runs()
best = best_run(runs, 'val_acc', 'max')
print(f'最优 run: lr={best["params"]["lr"]} layers={best["params"]["layers"]} '
      f'seed={best["params"]["seed"]}  val_acc={best["metrics"]["val_acc"]}')
# failed 的 0.640 虽不是最低，但必须被排除；最优应是 0.918（而非把 failed 算进来）
assert best['metrics']['val_acc'] == 0.918
assert best['params']['layers'] == 5
assert all(r['status']=='finished' for r in [best])
print('✅ 选优正确：排除 failed、口径统一、给出可复现的最优 run')

## 5 · config hash：复现性的廉价校验

给「params + 数据版本 + 代码版本」做规范化哈希得到 `config_hash`。两个用途：**去重**（同哈希=同配置）、**复现校验**（声称复现某 run，先比 config_hash）。

再加一个 `is_reproducible`：固定 seed 重训，结果应当一致。

In [ ]:
def config_hash(params, data_ver, code_ver):
    return canonical_hash({'params': params, 'data': data_ver, 'code': code_ver})

c1 = config_hash({'lr':0.01,'seed':42}, 'data@v1', 'code@c1d4e')
c2 = config_hash({'seed':42,'lr':0.01}, 'data@v1', 'code@c1d4e')   # 键序不同
c3 = config_hash({'lr':0.01,'seed':42}, 'data@v2', 'code@c1d4e')   # 数据版本变
assert c1 == c2, '同配置(键序无关)必得同哈希 -> 可去重'
assert c1 != c3, '数据版本变了，配置哈希必须变'
print('config_hash =', c1[:16], '...')

def train_model(seed):
    r = np.random.default_rng(seed)            # 显式 seed，杜绝隐藏随机性
    w = r.normal(size=8)
    return w

def is_reproducible(seed):
    return np.array_equal(train_model(seed), train_model(seed))

assert is_reproducible(42), '同 seed 必复现'
assert not np.array_equal(train_model(42), train_model(43)), '不同 seed 应不同'
print('✅ config_hash 去重 + 复现校验成立（一切随机性经显式 seed）')

---
## ✏️ 练习 1：按 tag 与 param 查询 run

实现 `query_runs(runs, where)`：返回所有 **同时满足** `where` 里全部条件的 run。
`where` 形如 `{'tags.owner': 'zerui', 'params.layers': 5, 'status': 'finished'}`——键用点号表示嵌套路径（`tags.owner` 即 `run['tags']['owner']`），值要相等。
缺失的路径视为不匹配（不报错）。

In [ ]:
def query_runs(runs, where):
    # TODO: 对每个 run，逐条检查 where 中的 'a.b'->run['a']['b'] 是否等于给定值；
    #       全部满足才保留。路径缺失则该 run 不匹配。返回匹配的 run 列表。
    #       提示：用 split('.') 沿 dict 逐层取值，缺键则跳过该 run。
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
pool = []
tk3 = Tracker(WORK, 'q')
for params, owner, st in [({'layers':5,'lr':0.01},'zerui','finished'),
                          ({'layers':3,'lr':0.01},'zerui','finished'),
                          ({'layers':5,'lr':0.02},'bob','finished'),
                          ({'layers':5,'lr':0.01},'zerui','failed')]:
    rr = tk3.start_run(params, tags={'owner':owner}); tk3.end_run(rr, st)
pool = tk3.all_runs()
res = query_runs(pool, {'tags.owner':'zerui','params.layers':5,'status':'finished'})
assert len(res) == 1, f'应只命中 1 个，得到 {len(res)}'
assert res[0]['params']['lr'] == 0.01 and res[0]['status']=='finished'
assert query_runs(pool, {'params.layers':5}) and len(query_runs(pool, {'params.layers':5}))==3
assert query_runs(pool, {'params.nonexist':1}) == []   # 缺路径 -> 不匹配
print('✅ 练习 1 通过：嵌套路径查询正确')

## ✏️ 练习 2：检测重复实验（靠 config hash 去重）

实现 `find_duplicates(runs, data_ver, code_ver)`：把 **config_hash 相同**（即 params 相同、且给定相同 data/code 版本）的 run 分到一组，返回 `{config_hash: [run_id, ...]}` 中 **长度 ≥ 2** 的组（即真正重复的）。

In [ ]:
def find_duplicates(runs, data_ver, code_ver):
    # TODO: 对每个 run 算 config_hash(run['params'], data_ver, code_ver)，
    #       按哈希聚合 run_id；只返回出现 >=2 次的哈希组。返回 dict。
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
tk4 = Tracker(WORK, 'dup')
ids = []
for params in [{'lr':0.01,'layers':5}, {'layers':5,'lr':0.01}, {'lr':0.02,'layers':5}]:
    rr = tk4.start_run(params); tk4.end_run(rr); ids.append(rr['run_id'])
dups = find_duplicates(tk4.all_runs(), 'data@v1', 'code@c1')
# 前两个 params 内容相同（仅键序不同）-> 应被判为重复
assert len(dups) == 1, f'应恰有 1 组重复，得到 {len(dups)}'
only_group = list(dups.values())[0]
assert set(only_group) == {ids[0], ids[1]}, '重复组应是前两个 run'
assert ids[2] not in only_group, '第三个 lr 不同，不算重复'
print('✅ 练习 2 通过：config hash 正确发现重复实验')

## ✏️ 练习 3：从指标历史诊断过拟合

`log_metric` 存了带 step 的曲线。实现 `is_overfitting(run, train_key, val_key, patience=2)`：
若 **验证指标连续 `patience` 步不再上升**（出现下降或持平），判为开始过拟合，返回 `True`，否则 `False`。
（只看 val 曲线即可；history 里每项是 `[step, value]` 列表。）

In [ ]:
def is_overfitting(run, val_key, patience=2):
    # TODO: 取 run['history'][val_key] 的 value 序列；从某点起连续 patience 步
    #       都 <= 之前出现过的最大值（不再创新高）则返回 True。
    #       提示：维护 running max 与「连续未创新高」计数器。
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
tk5 = Tracker(WORK, 'of')
ra = tk5.start_run({'m':'a'})
for s,v in enumerate([0.6,0.7,0.8,0.82,0.81,0.80]):    # 0.82 后连续 2 步下滑
    tk5.log_metric(ra, 'val_acc', v, step=s)
tk5.end_run(ra)
rb = tk5.start_run({'m':'b'})
for s,v in enumerate([0.6,0.7,0.8,0.85,0.9,0.92]):     # 一路上升
    tk5.log_metric(rb, 'val_acc', v, step=s)
tk5.end_run(rb)
assert is_overfitting(tk5.load(ra['run_id']), 'val_acc', patience=2) == True
assert is_overfitting(tk5.load(rb['run_id']), 'val_acc', patience=2) == False
print('✅ 练习 3 通过：能从曲线诊断过拟合（早停信号）')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def query_runs(runs, where):
    def get(run, path):
        cur = run
        for k in path.split('.'):
            if not isinstance(cur, dict) or k not in cur:
                return (False, None)
            cur = cur[k]
        return (True, cur)
    out = []
    for r in runs:
        ok = True
        for path, want in where.items():
            found, val = get(r, path)
            if not found or val != want:
                ok = False; break
        if ok: out.append(r)
    return out

In [ ]:
# 练习 2 参考答案
def find_duplicates(runs, data_ver, code_ver):
    groups = {}
    for r in runs:
        h = config_hash(r['params'], data_ver, code_ver)
        groups.setdefault(h, []).append(r['run_id'])
    return {h: ids for h, ids in groups.items() if len(ids) >= 2}

In [ ]:
# 练习 3 参考答案
def is_overfitting(run, val_key, patience=2):
    seq = [v for _, v in run['history'].get(val_key, [])]
    best = -float('inf'); stale = 0
    for v in seq:
        if v > best:
            best = v; stale = 0
        else:
            stale += 1
            if stale >= patience:
                return True
    return False

---
## 🧪 真实数据胶囊：用 `sqlite3` 建一个真正的查询索引

生产追踪器的真实架构是「**JSON/对象存储放大数据 + 关系数据库做索引查询**」（MLflow 正是如此）。

这里用标准库 `sqlite3` 把我们落盘的 run **索引进一张表**，然后用真正的 SQL 跑「按超参过滤 + 按指标排序选优」——你会看到第 4 节的 `best_run` 其实就是一句 `ORDER BY ... LIMIT 1`。

In [ ]:
import sqlite3

def build_index(runs, db_path):
    con = sqlite3.connect(db_path)
    con.execute('CREATE TABLE runs (run_id TEXT PRIMARY KEY, status TEXT, '
                'lr REAL, layers INTEGER, seed INTEGER, val_acc REAL)')
    for r in runs:
        p = r['params']; m = r['metrics']
        con.execute('INSERT OR REPLACE INTO runs VALUES (?,?,?,?,?,?)',
                    (r['run_id'], r['status'], p.get('lr'), p.get('layers'),
                     p.get('seed'), m.get('val_acc')))
    con.commit()
    return con

con = build_index(tk2.all_runs(), os.path.join(WORK, 'index.db'))
# 真正的 SQL：finished 中 layers=5 按 val_acc 选最优
row = con.execute("SELECT run_id, lr, layers, val_acc FROM runs "
                  "WHERE status='finished' AND layers=5 "
                  "ORDER BY val_acc DESC LIMIT 1").fetchone()
print('SQL 选优:', row)
assert abs(row[3] - 0.918) < 1e-9, 'SQL 选出的最优 val_acc 应为 0.918'
n_finished = con.execute("SELECT COUNT(*) FROM runs WHERE status='finished'").fetchone()[0]
assert n_finished == 4, 'tune 实验里有 4 个 finished run'
print('✅ sqlite 索引 + SQL 查询跑通：best_run == ORDER BY val_acc DESC LIMIT 1')

**🧪 胶囊练习**：用 SQL 实现「**超参重要性**」的一个最粗略版本——比较 `layers=5` 与 `layers=3` 两组 finished run 的 **平均 val_acc**，返回 `(avg_layers5, avg_layers3)`。

In [ ]:
def avg_acc_by_layers(con):
    # TODO: 用两条 SELECT AVG(val_acc) ... WHERE layers=? AND status='finished'
    #       返回 (平均@layers5, 平均@layers3)
    raise NotImplementedError

In [ ]:
# 自测
a5, a3 = avg_acc_by_layers(con)
assert a5 > a3, 'layers=5 这组平均更好（与构造一致）'
print(f'平均 val_acc: layers=5 -> {a5:.3f}, layers=3 -> {a3:.3f}')
print('✅ 胶囊练习通过：SQL 聚合给出粗略超参重要性')

In [ ]:
# 📖 胶囊参考答案
def avg_acc_by_layers(con):
    q = "SELECT AVG(val_acc) FROM runs WHERE layers=? AND status='finished'"
    a5 = con.execute(q, (5,)).fetchone()[0]
    a3 = con.execute(q, (3,)).fetchone()[0]
    return (a5, a3)

---
## 🔧 旁注：这套东西对应 MLflow 的什么

你刚写的 Tracker 几乎逐一对应 MLflow Tracking 的 API（伪代码，**本环境不跑**）：

```python
import mlflow
mlflow.set_experiment('tune_classifier')
with mlflow.start_run() as run:                 # == 我们的 start_run / end_run（上下文管理器自动 end）
    mlflow.log_params({'lr': 0.01, 'layers': 5}) # == log 到 run['params']
    for step, acc in enumerate(history):
        mlflow.log_metric('val_acc', acc, step=step)  # == log_metric 带 step（存曲线）
    mlflow.log_artifact('model.npz')             # == log_artifact（MLflow 也存内容/路径）
# 选优：mlflow.search_runs(order_by=['metrics.val_acc DESC'])  == 我们的 best_run / SQL
```

对应关系：`start_run`↔run 生命周期、`log_metric(step=)`↔曲线、`search_runs`↔我们的 query/best_run/SQL。MLflow 多出来的是：存储后端（S3/DB）、UI、模型注册、并发——但**数据模型与你写的一模一样**。

### 小结
- run = 一次实验的完整记录：params(输入) + metrics(输出，含曲线) + artifacts(产物+哈希) + tags(元数据)。
- **记录不可变**：finished 的 run 冻结，重实验另开新 run——这是可复现与可审计的前提。
- **可复现三层次**：配置可复现 < 种子可复现 < 字节级确定；至少记全 params + 固定显式 seed。
- **config hash** 规范化后哈希：去重 + 复现校验；选优只比 finished、统一口径。

下一站：**模块 02 · 数据与模型版本** —— 上线的到底是哪份数据、哪份权重？出事能回滚吗？